# Preproduccion — Evaluacion sobre datos no vistos

Esta es la primera y unica evaluacion de los modelos entrenados sobre el dataset de validacion.
Los artefactos se cargan desde `03_Modelos/01_Historial/` y se aplican usando exclusivamente `.transform()` — nunca `.fit()`.

**Objetivo**: confirmar que ambos modelos generalizan fuera de la muestra de entrenamiento y decidir cual es adecuado para produccion.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from pathlib import Path
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "README.md").exists():
    repo_root = repo_root.parent
if not (repo_root / "README.md").exists():
    raise FileNotFoundError("No se encontró la raíz del repositorio.")

## 1. Cargar datos de validacion

El dataset de validacion tiene nombres de columna inconsistentes con el entrenamiento y deben renombrarse antes de aplicar el pipeline.

In [ ]:
df = pd.read_csv(repo_root / "01_Datos/02_Validacion/validacion.csv")
df = df.rename(columns={
    "Lagging_Current_Reactive.Power_kVarh": "Lagging_Current_Reactive_Power_kVarh",
    "CO2(tCO2)": "CO2_tCO2",
})
print(f"Registros: {df.shape[0]} | Columnas: {df.shape[1]}")
print(df.columns.tolist())

## 2. Experimento A — Pipeline con variables fisicas

Carga los artefactos del Exp A y aplica el pipeline sobre los datos de validacion.
Las columnas categoricas (WeekStatus, Day_of_week, Load_Type) se transforman con el OHE entrenado.
Las 7 columnas numericas se escalan con el StandardScaler entrenado — incluyendo CO2_tCO2 como target estandarizado.

In [ ]:
hist = repo_root / "03_Modelos/01_Historial"
encoder_A = joblib.load(hist / "pipeline_encoder_A.joblib")
scaler_A  = joblib.load(hist / "pipeline_scaler_A.joblib")
modelo_A  = joblib.load(hist / "rfc_CO2_A_v1_pipeline.joblib")
print("Artefactos Exp A cargados.")

In [ ]:
cat_A = df[["WeekStatus", "Day_of_week", "Load_Type"]]
cat_ohe_A = pd.DataFrame(
    encoder_A.transform(cat_A),
    columns=encoder_A.get_feature_names_out(),
    index=df.index
)

num_cols_A = ["Usage_kWh", "Lagging_Current_Reactive_Power_kVarh",
              "Leading_Current_Reactive_Power_kVarh", "Lagging_Current_Power_Factor",
              "Leading_Current_Power_Factor", "NSM", "CO2_tCO2"]
num_std_A = pd.DataFrame(
    scaler_A.transform(df[num_cols_A]),
    columns=[c + "_std_" for c in num_cols_A],
    index=df.index
)

tablon_A = pd.concat([cat_ohe_A, num_std_A], axis=1)
X_val_A = tablon_A.drop(columns=["CO2_tCO2_std_"])
y_val_A = tablon_A["CO2_tCO2_std_"]

y_pred_A = modelo_A.predict(X_val_A)

r2_A   = r2_score(y_val_A, y_pred_A)
rmse_A = np.sqrt(mean_squared_error(y_val_A, y_pred_A))
mae_A  = mean_absolute_error(y_val_A, y_pred_A)

print(f"R²   : {r2_A:.6f}")
print(f"RMSE : {rmse_A:.6f}")
print(f"MAE  : {mae_A:.6f}")

## 3. Experimento B — Pipeline sin variables fisicas

El Exp B excluye Usage_kWh y Lagging_Current_Reactive_Power_kVarh. En su lugar usa features temporales extraidas de `date` y variables de eficiencia electrica.
Estas features son conocidas por el operador antes de que ocurra el consumo, lo que permite anticipacion operacional.

In [ ]:
encoder_B = joblib.load(hist / "pipeline_encoder_B.joblib")
scaler_B  = joblib.load(hist / "pipeline_scaler_B.joblib")
modelo_B  = joblib.load(hist / "rfc_CO2_B_v1_pipeline.joblib")

df["date"] = pd.to_datetime(df["date"], dayfirst=True)
df["hora"] = df["date"].dt.hour
df["turno"] = df["date"].dt.hour.map(
    lambda h: "Mañana" if 6 <= h < 14 else ("Tarde" if 14 <= h < 22 else "Noche")
)
df["es_fin_de_semana"] = (df["WeekStatus"] == "Weekend").astype(int)
df["NSM_sin"] = np.sin(2 * np.pi * df["NSM"] / 86400)
df["NSM_cos"] = np.cos(2 * np.pi * df["NSM"] / 86400)

print("Artefactos Exp B cargados. Features temporales generadas.")
print(df["turno"].value_counts())

In [ ]:
cat_B = df[["Day_of_week", "Load_Type", "turno"]]
cat_ohe_B = pd.DataFrame(
    encoder_B.transform(cat_B),
    columns=encoder_B.get_feature_names_out(),
    index=df.index
)

num_cols_B = ["Leading_Current_Reactive_Power_kVarh", "Lagging_Current_Power_Factor",
              "Leading_Current_Power_Factor", "NSM_sin", "NSM_cos", "hora", "CO2_tCO2"]
num_std_arr_B = scaler_B.transform(df[num_cols_B])
num_std_B = pd.DataFrame(
    num_std_arr_B,
    columns=[c + "_std_" for c in num_cols_B],
    index=df.index
)

tablon_B = pd.concat([cat_ohe_B, df[["es_fin_de_semana"]], num_std_B], axis=1)
y_val_B = num_std_arr_B[:, 6]
X_val_B = tablon_B.drop(columns=["CO2_tCO2_std_"])

y_pred_B = modelo_B.predict(X_val_B)

r2_B   = r2_score(y_val_B, y_pred_B)
rmse_B = np.sqrt(mean_squared_error(y_val_B, y_pred_B))
mae_B  = mean_absolute_error(y_val_B, y_pred_B)

print(f"R²   : {r2_B:.6f}")
print(f"RMSE : {rmse_B:.6f}")
print(f"MAE  : {mae_B:.6f}")

## 4. Tabla comparativa y diagnostico de overfitting

In [ ]:
ref = {
    "A": {"r2": 0.992343, "rmse": 0.08747, "mae": 0.00616},
    "B": {"r2": 0.846997, "rmse": 0.390998, "mae": 0.193321},
}

resultados = pd.DataFrame({
    "Experimento": ["A (con variables fisicas)", "B (sin variables fisicas)"],
    "R2 test":    [ref["A"]["r2"],  ref["B"]["r2"]],
    "R2 val":     [round(r2_A, 4),  round(r2_B, 4)],
    "RMSE test":  [ref["A"]["rmse"], ref["B"]["rmse"]],
    "RMSE val":   [round(rmse_A, 4), round(rmse_B, 4)],
    "Deg. RMSE":  [
        f"{(rmse_A - ref['A']['rmse']) / ref['A']['rmse'] * 100:.1f}%",
        f"{(rmse_B - ref['B']['rmse']) / ref['B']['rmse'] * 100:.1f}%",
    ],
    "Diagnostico": [
        "Generaliza" if (rmse_A - ref["A"]["rmse"]) / ref["A"]["rmse"] * 100 <= 10 else "Overfitting",
        "Generaliza" if (rmse_B - ref["B"]["rmse"]) / ref["B"]["rmse"] * 100 <= 10 else "Overfitting",
    ]
})
print(resultados.to_string(index=False))

## 5. Conclusion de negocio

Ambos modelos generalizan correctamente — en ninguno se observan senales de overfitting. Los valores de validacion son incluso ligeramente mejores que los de test, lo que indica que el conjunto de validacion representa bien la distribucion del problema.

**Experimento A — modelo de referencia para auditoria**
R2=0.9959 confirma la relacion fisica casi perfecta CO2-consumo. Es el modelo adecuado para reportes de huella de carbono cuando Usage_kWh esta disponible en tiempo real. No es util para anticipacion: conocer Usage_kWh del intervalo actual implica que el consumo ya ocurrio.

**Experimento B — modelo accionable para operacion**
R2=0.8745 en datos no vistos usando solo features temporales y de eficiencia electrica. Permite anticipar emisiones antes de que ocurran, habilitando decisiones preventivas: redistribucion de carga, ajuste de turno, programacion de mantenimiento. Suficiente para soporte operacional en produccion.

**Decision para produccion**: Exp B como modelo principal de la aplicacion. Exp A como pipeline de referencia para contextos de monitoreo y auditoria.